In [2]:
from pathlib import Path
import pandas as pd

# Works whether the notebook starts from the project root or /notebooks.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("\nMovieLens source files:")

for search_root in [DATA_ROOT / "raw"]:
    if search_root.exists():
        for path in sorted(search_root.rglob("*")):
            if path.is_file() and path.name.lower() in {
                "links.csv", "movies.csv"
            }:
                print(f"  {path.relative_to(PROJECT_ROOT)}")

print("\nPossible processed catalogue or mapping files:")

for folder_name in ["interim", "processed"]:
    folder = DATA_ROOT / folder_name
    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if not path.is_file():
                continue

            name = path.name.lower()
            relevant = (
                any(word in name for word in
                    ["mapping", "map", "catalog", "movie", "item", "index"])
                and path.suffix.lower() in {
                    ".csv", ".parquet", ".json", ".pkl", ".npy", ".npz"
                }
            )

            if relevant:
                print(f"  {path.relative_to(PROJECT_ROOT)}")

Project root: c:\Users\Lenovo\Documents\movie-recommender

MovieLens source files:
  data\raw\ml-32m\links.csv
  data\raw\ml-32m\movies.csv

Possible processed catalogue or mapping files:
  data\interim\debug_1pct\movies.parquet
  data\interim\sample_10pct\movies.parquet
  data\processed\sample_10pct\item_map.parquet
  data\processed\sample_10pct\user_map.parquet


In [3]:
import pandas as pd

links_path = DATA_ROOT / "raw" / "ml-32m" / "links.csv"
catalog_path = DATA_ROOT / "interim" / "sample_10pct" / "movies.parquet"
item_map_path = DATA_ROOT / "processed" / "sample_10pct" / "item_map.parquet"

links = pd.read_csv(links_path)
catalog = pd.read_parquet(catalog_path)
item_map = pd.read_parquet(item_map_path)

for name, df in [
    ("MovieLens links.csv", links),
    ("Sample catalogue", catalog),
    ("Item mapping", item_map),
]:
    print(f"\n{'=' * 60}")
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("Columns:", df.columns.tolist())
    print("\nData types:")
    print(df.dtypes.to_string())
    print("\nFirst 5 rows:")
    display(df.head())


MovieLens links.csv: 87,585 rows × 3 columns
Columns: ['movieId', 'imdbId', 'tmdbId']

Data types:
movieId      int64
imdbId       int64
tmdbId     float64

First 5 rows:


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0



Sample catalogue: 87,585 rows × 3 columns
Columns: ['movieId', 'title', 'genres']

Data types:
movieId    int64
title        str
genres       str

First 5 rows:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Item mapping: 17,241 rows × 2 columns
Columns: ['i', 'movieId']

Data types:
i          int32
movieId    int32

First 5 rows:


,i,movieId
0,0,1
1,1,2
2,2,3
3,3,4
4,4,5


# Audit TMDB ID coverage

In [4]:
import numpy as np
import pandas as pd

# Check the uniqueness of the source identifiers first.
print("Duplicate movieId rows in links:",
      int(links["movieId"].duplicated().sum()))

print("Duplicate movieId rows in item_map:",
      int(item_map["movieId"].duplicated().sum()))

print("Duplicate internal indices in item_map:",
      int(item_map["i"].duplicated().sum()))

# Ensure our joins cannot silently multiply rows.
if links["movieId"].duplicated().any():
    raise ValueError(
        "links.csv contains duplicate movieId values. "
        "Inspect these before continuing."
    )

if (
    item_map["movieId"].duplicated().any()
    or item_map["i"].duplicated().any()
):
    raise ValueError(
        "item_map contains duplicate movieId or internal index values."
    )

# Keep the internal index attached throughout the audit.
mapping = item_map.merge(
    links[["movieId", "imdbId", "tmdbId"]],
    on="movieId",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Attach titles for readable diagnostics.
mapping = mapping.merge(
    catalog[["movieId", "title"]],
    on="movieId",
    how="left",
    validate="one_to_one",
)

# Validate that available TMDB IDs are positive integers.
tmdb_numeric = pd.to_numeric(mapping["tmdbId"], errors="coerce")

valid_tmdb = (
    tmdb_numeric.notna()
    & tmdb_numeric.gt(0)
    & np.isclose(
        tmdb_numeric.fillna(0),
        np.round(tmdb_numeric.fillna(0)),
    )
)

matched_links = mapping["_merge"].eq("both")
missing_tmdb = ~valid_tmdb

# Detect TMDB IDs shared by multiple MovieLens movies.
tmdb_counts = (
    tmdb_numeric.loc[valid_tmdb]
    .value_counts()
)

duplicate_tmdb_ids = tmdb_counts[tmdb_counts > 1]
shared_id_rows = (
    valid_tmdb
    & tmdb_numeric.isin(duplicate_tmdb_ids.index)
)

print("\n" + "=" * 60)
print("TMDB MAPPING AUDIT")
print("=" * 60)

print(f"Model catalogue rows:          {len(item_map):,}")
print(f"Unique internal indices:       {item_map['i'].nunique():,}")
print(f"Unique MovieLens IDs:          {item_map['movieId'].nunique():,}")

print(f"\nRows matched in links.csv:      {matched_links.sum():,}")
print(f"Rows without a links match:     {(~matched_links).sum():,}")

print(f"\nValid positive integer TMDB IDs: {valid_tmdb.sum():,}")
print(f"Missing/invalid TMDB IDs:        {missing_tmdb.sum():,}")
print(f"Unique valid TMDB IDs:           {tmdb_numeric[valid_tmdb].nunique():,}")

coverage = 100 * valid_tmdb.mean()
print(f"Potential API lookup coverage:   {coverage:.2f}%")

print(f"\nTMDB IDs shared across movies:   {len(duplicate_tmdb_ids):,}")
print(f"Catalogue rows using shared IDs: {int(shared_id_rows.sum()):,}")

print("\nExamples without valid TMDB IDs:")
display(
    mapping.loc[missing_tmdb, ["i", "movieId", "title", "tmdbId"]]
    .head(15)
)

if len(duplicate_tmdb_ids):
    print("\nExamples of TMDB IDs shared by multiple MovieLens movies:")
    shared_examples = (
        mapping.loc[
            shared_id_rows,
            ["i", "movieId", "title", "tmdbId"],
        ]
        .sort_values("tmdbId")
        .head(20)
    )
    display(shared_examples)

Duplicate movieId rows in links: 0
Duplicate movieId rows in item_map: 0
Duplicate internal indices in item_map: 0

TMDB MAPPING AUDIT
Model catalogue rows:          17,241
Unique internal indices:       17,241
Unique MovieLens IDs:          17,241

Rows matched in links.csv:      17,241
Rows without a links match:     0

Valid positive integer TMDB IDs: 17,226
Missing/invalid TMDB IDs:        15
Unique valid TMDB IDs:           17,218
Potential API lookup coverage:   99.91%

TMDB IDs shared across movies:   8
Catalogue rows using shared IDs: 16

Examples without valid TMDB IDs:


,i,movieId,title,tmdbId
686,686,730,Low Life (1994),NaN
743,743,791,"Last Klezmer: Leopold Kozlowski, His Life and ...",NaN
1029,1029,1107,Loser (1991),NaN
1056,1056,1142,Get Over It (1996),NaN
1211,1211,1316,Anna (1996),NaN
1309,1309,1421,Grateful Dead (1995),NaN
1321,1321,1434,"Stranger, The (1994)",NaN
2575,2575,2851,Saturn 3 (1980),NaN
3639,3639,4051,Horrors of Spider Island (Ein Toter Hing im Ne...,NaN
5126,5126,5738,Faces of Death 5 (1996),NaN



Examples of TMDB IDs shared by multiple MovieLens movies:


,i,movieId,title,tmdbId
5343,5343,6003,Confessions of a Dangerous Mind (2002),4912.0
14020,14020,144606,Confessions of a Dangerous Mind (2002),4912.0
15143,15143,174533,Pokémon: Spell of the Unown (2000),10991.0
3808,3808,4241,Pokémon 3: The Movie (2001),10991.0
5074,5074,5672,Pokemon 4 Ever (a.k.a. Pokémon 4: The Movie) (...,12600.0
15145,15145,174543,Pokémon 4Ever: Celebi - Voice of the Forest (2...,12600.0
16138,16138,202599,Enron: The Smartest Guys in the Room (2005),13020.0
8262,8262,33154,Enron: The Smartest Guys in the Room (2005),13020.0
14013,14013,144426,Clockstoppers (2002),15028.0
4725,4725,5264,Clockstoppers (2002),15028.0


In [5]:
import time
import requests
import pandas as pd
from getpass import getpass

# Enter your TMDB API Read Access Token when prompted.
# It will not be printed or saved in the notebook source.
TMDB_TOKEN = getpass("TMDB Read Access Token: ").strip()

if not TMDB_TOKEN:
    raise ValueError("No token entered.")

# Select five unique TMDB IDs from our actual model catalogue.
pilot_candidates = mapping.loc[
    valid_tmdb,
    ["i", "movieId", "tmdbId", "title"]
].copy()

pilot_candidates["tmdbId"] = pd.to_numeric(
    pilot_candidates["tmdbId"], errors="raise"
).astype("int64")

pilot_candidates = (
    pilot_candidates
    .drop_duplicates("tmdbId")
    .sample(n=5, random_state=2026)
    .sort_values("i")
    .reset_index(drop=True)
)

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json",
})

pilot_results = []

for row in pilot_candidates.itertuples(index=False):
    url = f"https://api.themoviedb.org/3/movie/{row.tmdbId}"

    try:
        response = session.get(
            url,
            params={
                "append_to_response": "credits,keywords",
                "language": "en-US",
            },
            timeout=20,
        )
    except requests.RequestException as exc:
        pilot_results.append({
            "i": row.i,
            "movieId": row.movieId,
            "tmdbId": row.tmdbId,
            "catalog_title": row.title,
            "status": f"Request error: {type(exc).__name__}",
        })
        continue

    if response.status_code != 200:
        try:
            error_message = response.json().get(
                "status_message", "No error message returned"
            )
        except ValueError:
            error_message = "Non-JSON error response"

        pilot_results.append({
            "i": row.i,
            "movieId": row.movieId,
            "tmdbId": row.tmdbId,
            "catalog_title": row.title,
            "status": f"HTTP {response.status_code}: {error_message}",
        })
        continue

    data = response.json()

    credits = data.get("credits") or {}
    keyword_data = data.get("keywords") or {}

    keywords = (
        keyword_data.get("keywords")
        or keyword_data.get("results")
        or []
    )

    crew = credits.get("crew") or []
    cast = credits.get("cast") or []

    directors = [
        person.get("name")
        for person in crew
        if person.get("job") == "Director"
        and person.get("name")
    ]

    pilot_results.append({
        "i": row.i,
        "movieId": row.movieId,
        "tmdbId": row.tmdbId,
        "catalog_title": row.title,
        "tmdb_title": data.get("title"),
        "returned_id_matches": data.get("id") == row.tmdbId,
        "overview_chars": len(data.get("overview") or ""),
        "genre_count": len(data.get("genres") or []),
        "keyword_count": len(keywords),
        "cast_count": len(cast),
        "director_count": len(directors),
        "status": "OK",
    })

    time.sleep(0.25)

display(pd.DataFrame(pilot_results))

ValueError: No token entered.

In [5]:
import socket
import requests

# 1. Check whether your computer can resolve TMDB's hostname.
try:
    result = socket.getaddrinfo(
        "api.themoviedb.org", 443
    )
    print("DNS resolution: SUCCESS")
    print("Resolved addresses:", sorted({
        entry[4][0] for entry in result
    }))
except Exception as exc:
    print("DNS resolution: FAILED")
    print(type(exc).__name__, str(exc))

# 2. Test HTTPS connectivity without authentication.
# An HTTP 401 is acceptable here: it proves we reached the API.
try:
    response = requests.get(
        "https://api.themoviedb.org/3/configuration",
        timeout=15,
    )
    print("\nUnauthenticated HTTPS request: CONNECTED")
    print("HTTP status:", response.status_code)
except requests.RequestException as exc:
    print("\nUnauthenticated HTTPS request: FAILED")
    print(type(exc).__name__, str(exc))

# 3. Test the same endpoint with your existing token.
# Never print session.headers or TMDB_TOKEN.
try:
    response = session.get(
        "https://api.themoviedb.org/3/configuration",
        timeout=15,
    )
    print("\nAuthenticated HTTPS request: CONNECTED")
    print("HTTP status:", response.status_code)

    if response.status_code != 200:
        try:
            print(
                "TMDB error:",
                response.json().get(
                    "status_message",
                    "No error message returned",
                ),
            )
        except ValueError:
            print("Response was not JSON.")

except requests.RequestException as exc:
    print("\nAuthenticated HTTPS request: FAILED")
    print(type(exc).__name__, str(exc))

DNS resolution: SUCCESS
Resolved addresses: ['13.224.245.44', '13.224.245.47', '13.224.245.63', '13.224.245.92']

Unauthenticated HTTPS request: FAILED
ConnectionError ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

Authenticated HTTPS request: CONNECTED
HTTP status: 200


Try fetching just one movie details

In [6]:
import requests

row = pilot_candidates.iloc[0]

url = f"https://api.themoviedb.org/3/movie/{int(row['tmdbId'])}"

try:
    response = session.get(
        url,
        params={
            "append_to_response": "credits,keywords",
            "language": "en-US",
        },
        timeout=30,
    )

    print("HTTP status:", response.status_code)

    if response.status_code == 200:
        data = response.json()
        credits = data.get("credits") or {}
        keyword_data = data.get("keywords") or {}

        keywords = (
            keyword_data.get("keywords")
            or keyword_data.get("results")
            or []
        )

        print("Requested TMDB ID:", int(row["tmdbId"]))
        print("Returned TMDB ID:", data.get("id"))
        print("Catalogue title:", row["title"])
        print("TMDB title:", data.get("title"))
        print("Overview available:", bool(data.get("overview")))
        print("Genres:", len(data.get("genres") or []))
        print("Keywords:", len(keywords))
        print("Cast:", len(credits.get("cast") or []))
        print("Credits returned:", "credits" in data)
        print("Keywords returned:", "keywords" in data)

    else:
        try:
            print("TMDB message:", response.json().get("status_message"))
        except ValueError:
            print("Response was not JSON.")

except requests.RequestException as exc:
    print("Request failed:", type(exc).__name__, str(exc))

HTTP status: 200
Requested TMDB ID: 229
Returned TMDB ID: 229
Catalogue title: Bride of Frankenstein, The (Bride of Frankenstein) (1935)
TMDB title: Bride of Frankenstein
Overview available: True
Genres: 2
Keywords: 13
Cast: 55
Credits returned: True
Keywords returned: True


Retry the five-movie pilot with automatic retries

In [8]:
import time
import requests
import pandas as pd

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Retry temporary connection failures and recoverable HTTP statuses.
retry_policy = Retry(
    total=5,
    connect=5,
    read=2,
    status=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

adapter = HTTPAdapter(max_retries=retry_policy)
session.mount("https://", adapter)

pilot_rows = []

for row in pilot_candidates.itertuples(index=False):
    tmdb_id = int(row.tmdbId)

    try:
        response = session.get(
            f"https://api.themoviedb.org/3/movie/{tmdb_id}",
            params={
                "append_to_response": "credits,keywords",
                "language": "en-US",
            },
            timeout=(10, 30),
        )

        result = {
            "i": int(row.i),
            "movieId": int(row.movieId),
            "tmdbId": tmdb_id,
            "catalog_title": row.title,
            "http_status": response.status_code,
            "id_matches": False,
            "tmdb_title": None,
            "overview_chars": 0,
            "genres": 0,
            "keywords": 0,
            "cast": 0,
            "directors": 0,
            "status": "HTTP error",
        }

        if response.status_code == 200:
            movie = response.json()
            credits = movie.get("credits") or {}
            keyword_data = movie.get("keywords") or {}

            keywords = (
                keyword_data.get("keywords")
                or keyword_data.get("results")
                or []
            )

            directors = [
                person.get("name")
                for person in credits.get("crew", [])
                if person.get("job") == "Director"
                and person.get("name")
            ]

            result.update({
                "id_matches": movie.get("id") == tmdb_id,
                "tmdb_title": movie.get("title"),
                "overview_chars": len(movie.get("overview") or ""),
                "genres": len(movie.get("genres") or []),
                "keywords": len(keywords),
                "cast": len(credits.get("cast") or []),
                "directors": len(directors),
                "status": "OK",
            })
        else:
            try:
                result["status"] = response.json().get(
                    "status_message", "HTTP error"
                )
            except ValueError:
                result["status"] = "Non-JSON HTTP error"

        pilot_rows.append(result)

    except requests.RequestException as exc:
        pilot_rows.append({
            "i": int(row.i),
            "movieId": int(row.movieId),
            "tmdbId": tmdb_id,
            "catalog_title": row.title,
            "http_status": None,
            "id_matches": False,
            "tmdb_title": None,
            "overview_chars": 0,
            "genres": 0,
            "keywords": 0,
            "cast": 0,
            "directors": 0,
            "status": f"Request failed: {type(exc).__name__}",
        })

    time.sleep(0.25)

pilot_results = pd.DataFrame(pilot_rows)
display(pilot_results)

print("\nPilot summary")
print("Successful responses:", int((pilot_results["status"] == "OK").sum()), "/", len(pilot_results))
print("Correct ID matches:", int(pilot_results["id_matches"].sum()), "/", len(pilot_results))
print("Nonempty overviews:", int((pilot_results["overview_chars"] > 0).sum()), "/", len(pilot_results))

,i,movieId,tmdbId,catalog_title,http_status,id_matches,tmdb_title,overview_chars,genres,keywords,cast,directors,status
0,1233,1340,229,"Bride of Frankenstein, The (Bride of Frankenst...",200,True,Bride of Frankenstein,329,2,13,55,1,OK
1,4015,4475,15267,"Beast of War, The (Beast, The) (1988)",200,True,The Beast of War,179,2,2,21,1,OK
2,7380,25900,28436,"Curse of the Cat People, The (1944)",200,True,The Curse of the Cat People,133,3,15,11,2,OK
3,12139,100169,109520,After Porn Ends (2010),200,True,After Porn Ends,177,1,2,16,1,OK
4,16099,201444,565312,Knock Down the House (2019),200,True,Knock Down the House,348,1,6,8,1,OK



Pilot summary
Successful responses: 5 / 5
Correct ID matches: 5 / 5
Nonempty overviews: 5 / 5


In [9]:

import json
import time
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# 1. Define a cache directory under data/raw.
CACHE_DIR = DATA_ROOT / "raw" / "tmdb_api_cache"
CACHE_DIR_REL = CACHE_DIR.relative_to(PROJECT_ROOT).as_posix()

# Safety check: don't cache API data in a tracked directory.
ignore_probe = f"{CACHE_DIR_REL}/ignore_probe.json"
ignore_check = subprocess.run(
    ["git", "check-ignore", "-q", ignore_probe],
    cwd=PROJECT_ROOT,
    capture_output=True,
)

if ignore_check.returncode != 0:
    raise RuntimeError(
        f"{CACHE_DIR_REL} is not confirmed Git-ignored. "
        "Add an appropriate rule to .gitignore, then rerun."
    )

CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 2. Ensure we have an authenticated session.
# If the kernel restarted, enter the token securely.
if (
    "session" not in globals()
    or not session.headers.get("Authorization", "").startswith("Bearer ")
):
    from getpass import getpass

    token = getpass("TMDB Read Access Token: ").strip()
    if not token:
        raise ValueError("No token entered.")

    session = requests.Session()
    session.headers.update({
        "Authorization": f"Bearer {token}",
        "accept": "application/json",
    })

# Retry temporary failures, respecting server retry instructions.
retry_policy = Retry(
    total=5,
    connect=5,
    read=2,
    status=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retry_policy))

# 3. Find unique valid TMDB IDs from our audited mapping.
valid_id_values = pd.to_numeric(
    mapping.loc[valid_tmdb, "tmdbId"],
    errors="coerce",
).dropna().astype("int64")

unique_tmdb_ids = np.sort(valid_id_values.unique())

def valid_cached_response(tmdb_id):
    path = CACHE_DIR / f"{tmdb_id}.json"

    if not path.exists():
        return False

    try:
        with path.open("r", encoding="utf-8") as file:
            payload = json.load(file)

        return (
            payload.get("id") == int(tmdb_id)
            and isinstance(payload.get("credits"), dict)
            and isinstance(payload.get("keywords"), dict)
        )
    except (OSError, json.JSONDecodeError, AttributeError):
        return False

# Only select IDs that don't already have a valid cached response.
missing_ids = [
    int(tmdb_id)
    for tmdb_id in unique_tmdb_ids
    if not valid_cached_response(int(tmdb_id))
]

batch_size = min(100, len(missing_ids))
rng = np.random.default_rng(2026)
batch_ids = (
    sorted(rng.choice(missing_ids, size=batch_size, replace=False).tolist())
    if batch_size else []
)

print(f"Unique valid TMDB IDs: {len(unique_tmdb_ids):,}")
print(f"Already cached: {len(unique_tmdb_ids) - len(missing_ids):,}")
print(f"Missing from cache: {len(missing_ids):,}")
print(f"This batch: {len(batch_ids):,}")

# 4. Fetch and atomically save each valid response.
failures = []
successes = 0
fatal_auth_error = False

for position, tmdb_id in enumerate(batch_ids, start=1):
    cache_path = CACHE_DIR / f"{tmdb_id}.json"
    temp_path = CACHE_DIR / f".{tmdb_id}.json.tmp"

    try:
        response = session.get(
            f"https://api.themoviedb.org/3/movie/{tmdb_id}",
            params={
                "append_to_response": "credits,keywords",
                "language": "en-US",
            },
            timeout=(10, 30),
        )

        if response.status_code in (401, 403):
            failures.append({
                "tmdbId": tmdb_id,
                "status": response.status_code,
                "message": "Authentication/permission failure; batch stopped",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            })
            fatal_auth_error = True
            break

        if response.status_code != 200:
            try:
                message = response.json().get("status_message", "")
            except ValueError:
                message = "Non-JSON response"

            failures.append({
                "tmdbId": tmdb_id,
                "status": response.status_code,
                "message": message,
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            })
            continue

        payload = response.json()

        if (
            payload.get("id") != tmdb_id
            or not isinstance(payload.get("credits"), dict)
            or not isinstance(payload.get("keywords"), dict)
        ):
            failures.append({
                "tmdbId": tmdb_id,
                "status": "Incomplete/mismatched response",
                "message": "ID or appended fields failed validation",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            })
            continue

        # Write a temporary file, then replace it atomically.
        with temp_path.open("w", encoding="utf-8") as file:
            json.dump(payload, file, ensure_ascii=False)

        temp_path.replace(cache_path)
        successes += 1

    except requests.RequestException as exc:
        failures.append({
            "tmdbId": tmdb_id,
            "status": type(exc).__name__,
            "message": str(exc)[:300],
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        })

    finally:
        if temp_path.exists():
            temp_path.unlink()

    if position % 10 == 0 or position == len(batch_ids):
        print(
            f"Processed {position}/{len(batch_ids)} | "
            f"Saved this batch: {successes} | "
            f"Failures: {len(failures)}"
        )

    time.sleep(0.25)

# 5. Keep a record of failures so they can be reviewed and retried.
if failures:
    failure_path = CACHE_DIR / "fetch_failures.csv"
    new_failures = pd.DataFrame(failures)

    if failure_path.exists():
        old_failures = pd.read_csv(failure_path)
        new_failures = pd.concat(
            [old_failures, new_failures],
            ignore_index=True,
        )

    new_failures.to_csv(failure_path, index=False)

print("\nBatch summary")
print("Successful cached responses:", successes)
print("Failures recorded:", len(failures))
print("Stopped for authentication/permission:", fatal_auth_error)
print("Cache directory:", CACHE_DIR)
print(
    "Valid cached movies now:",
    sum(valid_cached_response(int(x)) for x in unique_tmdb_ids),
)


Unique valid TMDB IDs: 17,218
Already cached: 0
Missing from cache: 17,218
This batch: 100
Processed 10/100 | Saved this batch: 10 | Failures: 0
Processed 20/100 | Saved this batch: 20 | Failures: 0
Processed 30/100 | Saved this batch: 30 | Failures: 0
Processed 40/100 | Saved this batch: 40 | Failures: 0
Processed 50/100 | Saved this batch: 50 | Failures: 0
Processed 60/100 | Saved this batch: 60 | Failures: 0
Processed 70/100 | Saved this batch: 70 | Failures: 0
Processed 80/100 | Saved this batch: 80 | Failures: 0
Processed 90/100 | Saved this batch: 90 | Failures: 0
Processed 100/100 | Saved this batch: 100 | Failures: 0

Batch summary
Successful cached responses: 100
Failures recorded: 0
Stopped for authentication/permission: False
Cache directory: c:\Users\Lenovo\Documents\movie-recommender\data\raw\tmdb_api_cache
Valid cached movies now: 100


In [13]:

import json
import time
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# -------------------------------------------------------
# Configuration
# -------------------------------------------------------
BATCH_LIMIT = 1000
PAUSE_SECONDS = 0.25

CACHE_DIR = DATA_ROOT / "raw" / "tmdb_api_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_PATH = CACHE_DIR / "download_progress.json"
FAILURE_PATH = CACHE_DIR / "fetch_failures.csv"

# Confirm the cache stays out of Git.
probe = CACHE_DIR.relative_to(PROJECT_ROOT).as_posix() + "/ignore_probe.json"
check = subprocess.run(
    ["git", "check-ignore", "-q", probe],
    cwd=PROJECT_ROOT,
    capture_output=True,
)

if check.returncode != 0:
    raise RuntimeError(
        "The TMDB cache is not confirmed Git-ignored. "
        "Update .gitignore before continuing."
    )

# Reuse the authenticated session; ask securely if the kernel restarted.
if (
    "session" not in globals()
    or not session.headers.get("Authorization", "").startswith("Bearer ")
):
    from getpass import getpass

    token = getpass("TMDB Read Access Token: ").strip()
    if not token:
        raise ValueError("No token entered.")

    session = requests.Session()
    session.headers.update({
        "Authorization": f"Bearer {token}",
        "accept": "application/json",
    })

retry_policy = Retry(
    total=5,
    connect=5,
    read=2,
    status=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retry_policy))

# -------------------------------------------------------
# Discover unique valid IDs
# -------------------------------------------------------
tmdb_numeric = pd.to_numeric(mapping["tmdbId"], errors="coerce")

valid_id_mask = (
    tmdb_numeric.notna()
    & tmdb_numeric.gt(0)
    & np.isclose(
        tmdb_numeric.fillna(0),
        np.round(tmdb_numeric.fillna(0)),
    )
)

unique_tmdb_ids = sorted(
    tmdb_numeric.loc[valid_id_mask].astype("int64").unique().tolist()
)

def is_valid_cache(tmdb_id):
    path = CACHE_DIR / f"{int(tmdb_id)}.json"
    if not path.exists():
        return False

    try:
        with path.open("r", encoding="utf-8") as file:
            payload = json.load(file)

        return (
            payload.get("id") == int(tmdb_id)
            and isinstance(payload.get("credits"), dict)
            and isinstance(payload.get("keywords"), dict)
        )
    except (OSError, json.JSONDecodeError, AttributeError):
        return False

if PROGRESS_PATH.exists():
    with PROGRESS_PATH.open("r", encoding="utf-8") as file:
        progress = json.load(file)
else:
    progress = {"last_tmdb_id": 0}

last_tmdb_id = int(progress.get("last_tmdb_id", 0))

# Continue after the last attempted ID, skipping already cached IDs.
queue = [
    int(tmdb_id)
    for tmdb_id in unique_tmdb_ids
    if int(tmdb_id) > last_tmdb_id
    and not is_valid_cache(tmdb_id)
]

batch_ids = queue[:BATCH_LIMIT]

print(f"Unique IDs: {len(unique_tmdb_ids):,}")
print(f"Already validly cached: "
      f"{sum(is_valid_cache(x) for x in unique_tmdb_ids):,}")
print(f"Progress cursor: {last_tmdb_id:,}")
print(f"Uncached IDs after cursor: {len(queue):,}")
print(f"This batch: {len(batch_ids):,}")

if not batch_ids:
    print(
        "No new IDs after the progress cursor. "
        "Once the main queue is exhausted, audit and retry recorded failures."
    )

# -------------------------------------------------------
# Fetch and cache this batch
# -------------------------------------------------------
failures = []
successes = 0
fatal_auth_error = False
last_completed_id = last_tmdb_id

for position, tmdb_id in enumerate(batch_ids, start=1):
    cache_path = CACHE_DIR / f"{tmdb_id}.json"
    temp_path = CACHE_DIR / f".{tmdb_id}.json.tmp"

    try:
        response = session.get(
            f"https://api.themoviedb.org/3/movie/{tmdb_id}",
            params={
                "append_to_response": "credits,keywords",
                "language": "en-US",
            },
            timeout=(10, 30),
        )

        if response.status_code in (401, 403):
            failures.append({
                "tmdbId": tmdb_id,
                "status": response.status_code,
                "message": "Authentication/permission error; stopped batch",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            })
            fatal_auth_error = True
            # Do not advance the cursor past this ID.
            break

        if response.status_code != 200:
            try:
                message = response.json().get("status_message", "")
            except ValueError:
                message = "Non-JSON response"

            failures.append({
                "tmdbId": tmdb_id,
                "status": response.status_code,
                "message": message,
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            })
            last_completed_id = tmdb_id
            continue

        payload = response.json()

        if (
            payload.get("id") != tmdb_id
            or not isinstance(payload.get("credits"), dict)
            or not isinstance(payload.get("keywords"), dict)
        ):
            failures.append({
                "tmdbId": tmdb_id,
                "status": "Invalid response",
                "message": "ID mismatch or missing appended fields",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            })
            last_completed_id = tmdb_id
            continue

        with temp_path.open("w", encoding="utf-8") as file:
            json.dump(payload, file, ensure_ascii=False)

        temp_path.replace(cache_path)
        successes += 1
        last_completed_id = tmdb_id

    except requests.RequestException as exc:
        failures.append({
            "tmdbId": tmdb_id,
            "status": type(exc).__name__,
            "message": str(exc)[:300],
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        })
        last_completed_id = tmdb_id

    finally:
        if temp_path.exists():
            temp_path.unlink()

    if position % 100 == 0 or position == len(batch_ids):
        print(
            f"Processed {position}/{len(batch_ids)} | "
            f"successes={successes} | failures={len(failures)}"
        )

    time.sleep(PAUSE_SECONDS)

# Record failures without losing earlier failure history.
if failures:
    new_failures = pd.DataFrame(failures)

    if FAILURE_PATH.exists():
        old_failures = pd.read_csv(FAILURE_PATH)
        new_failures = pd.concat(
            [old_failures, new_failures],
            ignore_index=True,
        )

    new_failures.to_csv(FAILURE_PATH, index=False)

# Atomically save progress so the next execution can resume.
progress = {
    "last_tmdb_id": last_completed_id,
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
}
progress_temp = CACHE_DIR / ".download_progress.json.tmp"

with progress_temp.open("w", encoding="utf-8") as file:
    json.dump(progress, file, indent=2)

progress_temp.replace(PROGRESS_PATH)

valid_cached_count = sum(
    is_valid_cache(x) for x in unique_tmdb_ids
)

print("\nBatch summary")
print("Successful responses cached:", successes)
print("Failures recorded this batch:", len(failures))
print("Authentication/permission stop:", fatal_auth_error)
print("Valid cached movies:", valid_cached_count)
print("Progress cursor:", last_completed_id)
print("Remaining IDs after cursor:", sum(
    1 for x in unique_tmdb_ids
    if x > last_completed_id and not is_valid_cache(x)
))


Unique IDs: 17,218
Already validly cached: 1,097
Progress cursor: 1,698
Uncached IDs after cursor: 16,118
This batch: 1,000
Processed 100/1000 | successes=100 | failures=0
Processed 200/1000 | successes=200 | failures=0
Processed 300/1000 | successes=300 | failures=0
Processed 400/1000 | successes=398 | failures=2
Processed 500/1000 | successes=498 | failures=2
Processed 600/1000 | successes=598 | failures=2
Processed 700/1000 | successes=698 | failures=2
Processed 800/1000 | successes=798 | failures=2
Processed 900/1000 | successes=898 | failures=2
Processed 1000/1000 | successes=998 | failures=2

Batch summary
Successful responses cached: 998
Failures recorded this batch: 2
Authentication/permission stop: False
Valid cached movies: 2095
Progress cursor: 6414
Remaining IDs after cursor: 15118


In [14]:

import json
import time
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# -------------------------------------------------------
# Configuration
# -------------------------------------------------------
BATCH_SIZE = 1000
CHECKPOINT_EVERY = 100
PAUSE_SECONDS = 0.25

CACHE_DIR = DATA_ROOT / "raw" / "tmdb_api_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_PATH = CACHE_DIR / "download_progress.json"
FAILURE_PATH = CACHE_DIR / "fetch_failures.csv"

# Safety: make sure cache files will not be committed to Git.
probe = CACHE_DIR.relative_to(PROJECT_ROOT).as_posix() + "/ignore_probe.json"

check = subprocess.run(
    ["git", "check-ignore", "-q", probe],
    cwd=PROJECT_ROOT,
    capture_output=True,
)

if check.returncode != 0:
    raise RuntimeError(
        "TMDB cache is not confirmed Git-ignored. "
        "Fix .gitignore before continuing."
    )

# -------------------------------------------------------
# Authenticated session
# -------------------------------------------------------
if (
    "session" not in globals()
    or not session.headers.get("Authorization", "").startswith("Bearer ")
):
    token = getpass("TMDB Read Access Token: ").strip()
    if not token:
        raise ValueError("No token entered.")

    session = requests.Session()
    session.headers.update({
        "Authorization": f"Bearer {token}",
        "accept": "application/json",
    })

retry_policy = Retry(
    total=5,
    connect=5,
    read=2,
    status=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retry_policy))

# -------------------------------------------------------
# IDs and existing cache
# -------------------------------------------------------
tmdb_numeric = pd.to_numeric(mapping["tmdbId"], errors="coerce")

valid_mask = (
    tmdb_numeric.notna()
    & tmdb_numeric.gt(0)
    & np.isclose(
        tmdb_numeric.fillna(0),
        np.round(tmdb_numeric.fillna(0)),
    )
)

unique_ids = sorted(
    tmdb_numeric.loc[valid_mask].astype("int64").unique().tolist()
)

def is_valid_cache(tmdb_id):
    path = CACHE_DIR / f"{int(tmdb_id)}.json"

    if not path.exists():
        return False

    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        return (
            payload.get("id") == int(tmdb_id)
            and isinstance(payload.get("credits"), dict)
            and isinstance(payload.get("keywords"), dict)
        )
    except (OSError, ValueError, AttributeError):
        return False

if PROGRESS_PATH.exists():
    with PROGRESS_PATH.open("r", encoding="utf-8") as f:
        progress = json.load(f)
else:
    progress = {"last_tmdb_id": 0}

cursor = int(progress.get("last_tmdb_id", 0))

# Only queue uncached IDs beyond the saved cursor.
queue = [
    tmdb_id for tmdb_id in unique_ids
    if tmdb_id > cursor and not is_valid_cache(tmdb_id)
]

print(f"Unique TMDB IDs: {len(unique_ids):,}")
print(f"Currently valid cached: "
      f"{sum(is_valid_cache(x) for x in unique_ids):,}")
print(f"Starting after TMDB ID: {cursor}")
print(f"IDs queued for this run: {len(queue):,}")

# -------------------------------------------------------
# Checkpoint and failure helpers
# -------------------------------------------------------
def save_progress(last_id):
    state = {
        "last_tmdb_id": int(last_id),
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    temp = CACHE_DIR / ".download_progress.json.tmp"

    with temp.open("w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)

    temp.replace(PROGRESS_PATH)

def save_failures(rows):
    if not rows:
        return

    pd.DataFrame(rows).to_csv(
        FAILURE_PATH,
        mode="a",
        header=not FAILURE_PATH.exists(),
        index=False,
    )
    rows.clear()

# -------------------------------------------------------
# Automatically process every remaining batch
# -------------------------------------------------------
total_success = 0
total_failures = 0
processed = 0
failure_buffer = []
stop_reason = None
last_completed_id = cursor

try:
    for batch_start in range(0, len(queue), BATCH_SIZE):
        batch = queue[batch_start:batch_start + BATCH_SIZE]
        batch_success = 0
        batch_failures = 0

        print(
            f"\nStarting batch "
            f"{batch_start // BATCH_SIZE + 1} | "
            f"{len(batch)} IDs"
        )

        for tmdb_id in batch:
            cache_path = CACHE_DIR / f"{tmdb_id}.json"
            temp_path = CACHE_DIR / f".{tmdb_id}.json.tmp"

            try:
                response = session.get(
                    f"https://api.themoviedb.org/3/movie/{tmdb_id}",
                    params={
                        "append_to_response": "credits,keywords",
                        "language": "en-US",
                    },
                    timeout=(10, 30),
                )

                # Stop without advancing past this ID.
                # It can be retried after the issue is resolved.
                if response.status_code in (401, 403, 429):
                    stop_reason = f"HTTP {response.status_code}"
                    failure_buffer.append({
                        "tmdbId": tmdb_id,
                        "status": response.status_code,
                        "message": "Stopped for auth/rate-limit response",
                        "timestamp_utc": datetime.now(
                            timezone.utc
                        ).isoformat(),
                    })
                    break

                if response.status_code != 200:
                    try:
                        message = response.json().get(
                            "status_message", ""
                        )
                    except ValueError:
                        message = "Non-JSON response"

                    failure_buffer.append({
                        "tmdbId": tmdb_id,
                        "status": response.status_code,
                        "message": message,
                        "timestamp_utc": datetime.now(
                            timezone.utc
                        ).isoformat(),
                    })
                    batch_failures += 1
                    total_failures += 1
                    last_completed_id = tmdb_id

                else:
                    payload = response.json()

                    if (
                        payload.get("id") != tmdb_id
                        or not isinstance(payload.get("credits"), dict)
                        or not isinstance(payload.get("keywords"), dict)
                    ):
                        failure_buffer.append({
                            "tmdbId": tmdb_id,
                            "status": "Invalid response",
                            "message": "ID mismatch or missing appended data",
                            "timestamp_utc": datetime.now(
                                timezone.utc
                            ).isoformat(),
                        })
                        batch_failures += 1
                        total_failures += 1
                        last_completed_id = tmdb_id

                    else:
                        # Atomically save each successful response.
                        with temp_path.open("w", encoding="utf-8") as f:
                            json.dump(
                                payload,
                                f,
                                ensure_ascii=False,
                            )

                        temp_path.replace(cache_path)

                        batch_success += 1
                        total_success += 1
                        last_completed_id = tmdb_id

            except requests.RequestException as exc:
                failure_buffer.append({
                    "tmdbId": tmdb_id,
                    "status": type(exc).__name__,
                    "message": str(exc)[:300],
                    "timestamp_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                })
                batch_failures += 1
                total_failures += 1
                last_completed_id = tmdb_id

            finally:
                if temp_path.exists():
                    temp_path.unlink()

            processed += 1

            if processed % CHECKPOINT_EVERY == 0:
                save_failures(failure_buffer)
                save_progress(last_completed_id)

                print(
                    f"Total processed: {processed:,}/{len(queue):,} | "
                    f"Successes: {total_success:,} | "
                    f"Failures: {total_failures:,} | "
                    f"Last ID: {last_completed_id}"
                )

            time.sleep(PAUSE_SECONDS)

        # Save progress at every batch boundary too.
        save_failures(failure_buffer)
        save_progress(last_completed_id)

        print(
            f"Batch complete | "
            f"Successes: {batch_success} | "
            f"Failures: {batch_failures} | "
            f"Last ID: {last_completed_id}"
        )

        if stop_reason:
            break

except KeyboardInterrupt:
    save_failures(failure_buffer)
    save_progress(last_completed_id)
    print("\nInterrupted safely. Saved responses and progress.")

# -------------------------------------------------------
# Final status
# -------------------------------------------------------
save_failures(failure_buffer)
save_progress(last_completed_id)

cached_total = sum(is_valid_cache(x) for x in unique_ids)
uncached_total = len(unique_ids) - cached_total

print("\n" + "=" * 55)
print("DOWNLOAD STATUS")
print("=" * 55)
print(f"New successful responses: {total_success:,}")
print(f"Failures during this run: {total_failures:,}")
print(f"Valid cached responses overall: {cached_total:,}/{len(unique_ids):,}")
print(f"Still uncached: {uncached_total:,}")
print(f"Last progress cursor: {last_completed_id}")
print(f"Stop reason: {stop_reason or 'Queue finished'}")
print(f"Cache: {CACHE_DIR}")


Unique TMDB IDs: 17,218
Currently valid cached: 2,095
Starting after TMDB ID: 6414
IDs queued for this run: 15,118

Starting batch 1 | 1000 IDs
Total processed: 100/15,118 | Successes: 100 | Failures: 0 | Last ID: 7303
Total processed: 200/15,118 | Successes: 200 | Failures: 0 | Last ID: 8066
Total processed: 300/15,118 | Successes: 300 | Failures: 0 | Last ID: 8422
Total processed: 400/15,118 | Successes: 400 | Failures: 0 | Last ID: 8860
Total processed: 500/15,118 | Successes: 500 | Failures: 0 | Last ID: 9043
Total processed: 600/15,118 | Successes: 600 | Failures: 0 | Last ID: 9320
Total processed: 700/15,118 | Successes: 700 | Failures: 0 | Last ID: 9451
Total processed: 800/15,118 | Successes: 800 | Failures: 0 | Last ID: 9586
Total processed: 900/15,118 | Successes: 900 | Failures: 0 | Last ID: 9719
Total processed: 1,000/15,118 | Successes: 999 | Failures: 1 | Last ID: 9877
Batch complete | Successes: 999 | Failures: 1 | Last ID: 9877

Starting batch 2 | 1000 IDs
Total process

In [11]:
failure_path = CACHE_DIR / "fetch_failures.csv"

if failure_path.exists():
    failure_log = pd.read_csv(failure_path)

    print("Failure records:", len(failure_log))
    print("\nFailure types:")
    display(
        failure_log["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )

    print("\nRecorded failure details:")
    display(failure_log.tail(10))
else:
    print("No failure log exists.")

Failure records: 3

Failure types:


,status,count
0,404,2
1,ConnectionError,1



Recorded failure details:


,tmdbId,status,message,timestamp_utc
0,2,ConnectionError,"HTTPSConnectionPool(host='api.themoviedb.org',...",2026-10-12T00:47:12.762436+00:00
1,30,404,The resource you requested could not be found.,2026-10-12T00:47:24.217891+00:00
2,32,404,The resource you requested could not be found.,2026-10-12T00:47:24.463857+00:00


In [12]:
problem_ids = [2, 30, 32]

display(
    mapping.loc[
        pd.to_numeric(mapping["tmdbId"], errors="coerce").isin(problem_ids),
        ["i", "movieId", "title", "tmdbId", "imdbId"]
    ].sort_values("tmdbId")
)

,i,movieId,title,tmdbId,imdbId
4011,4011,4470,Ariel (1988),2.0,94675
14394,14394,158092,Magnetic Rose (1995),30.0,1530535
15680,15680,189483,Cannon Fodder (1995),32.0,6264824


In [7]:
import json
import numpy as np
import pandas as pd

CACHE_DIR = DATA_ROOT / "raw" / "tmdb_api_cache"
FAILURE_PATH = CACHE_DIR / "fetch_failures.csv"

# Reconstruct the complete set of valid MovieLens-to-TMDB IDs.
tmdb_numeric = pd.to_numeric(mapping["tmdbId"], errors="coerce")

valid_mask = (
    tmdb_numeric.notna()
    & tmdb_numeric.gt(0)
    & np.isclose(
        tmdb_numeric.fillna(0),
        np.round(tmdb_numeric.fillna(0)),
    )
)

all_tmdb_ids = sorted(
    tmdb_numeric.loc[valid_mask].astype("int64").unique().tolist()
)

def is_valid_cache(tmdb_id):
    path = CACHE_DIR / f"{int(tmdb_id)}.json"

    if not path.exists():
        return False

    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        return (
            payload.get("id") == int(tmdb_id)
            and isinstance(payload.get("credits"), dict)
            and isinstance(payload.get("keywords"), dict)
        )
    except (OSError, json.JSONDecodeError, AttributeError):
        return False

# Find every missing or invalid response, regardless of progress cursor.
uncached_ids = [
    tmdb_id for tmdb_id in all_tmdb_ids
    if not is_valid_cache(tmdb_id)
]

print(f"Unique TMDB IDs: {len(all_tmdb_ids):,}")
print(f"Valid cached responses: {len(all_tmdb_ids) - len(uncached_ids):,}")
print(f"Missing/invalid responses: {len(uncached_ids):,}")

# Load the historical failure log.
if FAILURE_PATH.exists():
    failure_log = pd.read_csv(FAILURE_PATH)

    failure_log["tmdbId"] = pd.to_numeric(
        failure_log["tmdbId"], errors="coerce"
    )

    print("\nHistorical failure records:", len(failure_log))
    print("\nFailure statuses:")
    display(
        failure_log["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )

    # Summarize potentially repeated failures per TMDB ID.
    failure_summary = (
        failure_log.dropna(subset=["tmdbId"])
        .groupby("tmdbId")
        .agg(
            attempts=("status", "size"),
            statuses=(
                "status",
                lambda s: ", ".join(map(str, pd.unique(s))),
            ),
        )
        .reset_index()
    )
else:
    failure_summary = pd.DataFrame(
        columns=["tmdbId", "attempts", "statuses"]
    )
    print("\nNo failure log found.")

# Attach MovieLens titles to every missing ID.
missing_report = (
    mapping.loc[valid_mask, ["tmdbId", "movieId", "title", "i"]]
    .assign(
        tmdbId=lambda df: pd.to_numeric(df["tmdbId"]).astype("int64")
    )
    .drop_duplicates("tmdbId")
    .loc[lambda df: df["tmdbId"].isin(uncached_ids)]
    .merge(failure_summary, on="tmdbId", how="left")
    .sort_values("tmdbId")
    .reset_index(drop=True)
)

missing_report["attempts"] = (
    missing_report["attempts"].fillna(0).astype(int)
)

print("\nMissing IDs with no logged failure:",
      int((missing_report["attempts"] == 0).sum()))

print("\nFirst 30 missing records:")
display(missing_report.head(30))

print("\nMissing records by logged status:")
display(
    missing_report.groupby("statuses", dropna=False)
    .size()
    .rename("movies")
    .reset_index()
    .sort_values("movies", ascending=False)
)


Unique TMDB IDs: 17,218
Valid cached responses: 17,035
Missing/invalid responses: 183

Historical failure records: 183

Failure statuses:


,status,count
0,404,182
1,ConnectionError,1



Missing IDs with no logged failure: 0

First 30 missing records:


,tmdbId,movieId,title,i,attempts,statuses
0,2,4470,Ariel (1988),4011,1,ConnectionError
1,30,158092,Magnetic Rose (1995),14394,1,404
2,32,189483,Cannon Fodder (1995),15680,1,404
3,2413,26379,Jesus of Nazareth (1977),7591,1,404
4,2518,27724,Hitler: The Rise of Evil (2003),7902,1,404
5,9795,65359,Earthsea (Legend of Earthsea) (2004),10104,1,404
6,10700,38198,Darwin's Nightmare (2004),8496,1,404
7,11343,31502,Salem's Lot (1979),8070,1,404
8,11788,84663,Elizabeth I (2005),11245,1,404
9,11944,27180,Joan of Arc (1999),7795,1,404



Missing records by logged status:


,statuses,movies
0,404,182
1,ConnectionError,1


In [8]:
import json
import time
import pandas as pd
import requests
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

CACHE_DIR = DATA_ROOT / "raw" / "tmdb_api_cache"
FAILURE_PATH = CACHE_DIR / "fetch_failures.csv"

# Reuse the authenticated session if it exists.
if (
    "session" not in globals()
    or not session.headers.get("Authorization", "").startswith("Bearer ")
):
    token = getpass("TMDB Read Access Token: ").strip()

    if not token:
        raise ValueError("No TMDB token supplied.")

    session = requests.Session()
    session.headers.update({
        "Authorization": f"Bearer {token}",
        "accept": "application/json",
    })

# Find IDs with recoverable connection failures.
failure_log = pd.read_csv(FAILURE_PATH)

recoverable_statuses = {
    "ConnectionError",
    "Timeout",
    "ConnectTimeout",
    "ReadTimeout",
}

failure_log["tmdbId"] = pd.to_numeric(
    failure_log["tmdbId"], errors="coerce"
)

retry_ids = sorted(
    failure_log.loc[
        failure_log["status"].isin(recoverable_statuses),
        "tmdbId",
    ].dropna().astype(int).unique().tolist()
)

# Never refetch a response that is already valid.
retry_ids = [
    tmdb_id for tmdb_id in retry_ids
    if not is_valid_cache(tmdb_id)
]

print("Recoverable IDs to retry:", retry_ids)

for tmdb_id in retry_ids:
    try:
        response = session.get(
            f"https://api.themoviedb.org/3/movie/{tmdb_id}",
            params={
                "append_to_response": "credits,keywords",
                "language": "en-US",
            },
            timeout=(10, 30),
        )

        if response.status_code == 200:
            payload = response.json()

            if (
                payload.get("id") == tmdb_id
                and isinstance(payload.get("credits"), dict)
                and isinstance(payload.get("keywords"), dict)
            ):
                temp_path = CACHE_DIR / f".{tmdb_id}.json.tmp"
                cache_path = CACHE_DIR / f"{tmdb_id}.json"

                with temp_path.open("w", encoding="utf-8") as f:
                    json.dump(payload, f, ensure_ascii=False)

                temp_path.replace(cache_path)
                print(f"{tmdb_id}: retry successful")
            else:
                print(f"{tmdb_id}: response failed validation")

        else:
            print(f"{tmdb_id}: HTTP {response.status_code}")

            pd.DataFrame([{
                "tmdbId": tmdb_id,
                "status": response.status_code,
                "message": "Retry attempt",
                "timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            }]).to_csv(
                FAILURE_PATH,
                mode="a",
                header=False,
                index=False,
            )

    except requests.RequestException as exc:
        print(f"{tmdb_id}: {type(exc).__name__}")

    time.sleep(0.3)

# Final cache coverage audit.
all_ids = sorted(
    pd.to_numeric(mapping["tmdbId"], errors="coerce")
    .dropna()
    .loc[lambda s: (s > 0) & (s % 1 == 0)]
    .astype(int)
    .unique()
)

valid_count = sum(is_valid_cache(tmdb_id) for tmdb_id in all_ids)
missing_ids = [
    tmdb_id for tmdb_id in all_ids
    if not is_valid_cache(tmdb_id)
]

print("\nFINAL CACHE AUDIT")
print(f"Valid cached IDs: {valid_count:,}/{len(all_ids):,}")
print(f"Coverage: {100 * valid_count / len(all_ids):.2f}%")
print(f"Still unresolved: {len(missing_ids):,}")

Recoverable IDs to retry: [2]
2: retry successful

FINAL CACHE AUDIT
Valid cached IDs: 17,036/17,218
Coverage: 98.94%
Still unresolved: 182


In [9]:
# TMDB enrichment: parse existing cache only.
# IMPORTANT: This cell makes NO API requests.

import json
import pandas as pd
from pathlib import Path

TMDB_CACHE_DIR = DATA_ROOT / "raw" / "tmdb_api_cache"

# Reconstruct unique, valid TMDB IDs from the existing mapping.
tmdb_numeric = pd.to_numeric(mapping["tmdbId"], errors="coerce")

valid_id_mask = (
    tmdb_numeric.notna()
    & tmdb_numeric.gt(0)
    & (tmdb_numeric % 1 == 0)
)

unique_tmdb_ids = sorted(
    tmdb_numeric.loc[valid_id_mask].astype("int64").unique()
)

def get_names(items):
    """Extract non-empty names from TMDB list-of-dictionary fields."""
    if not isinstance(items, list):
        return []

    return [
        item["name"].strip()
        for item in items
        if isinstance(item, dict)
        and isinstance(item.get("name"), str)
        and item["name"].strip()
    ]

# One record per unique TMDB ID, including unresolved IDs.
tmdb_records = []

for tmdb_id in unique_tmdb_ids:
    cache_path = TMDB_CACHE_DIR / f"{tmdb_id}.json"

    record = {
        "tmdbId_key": tmdb_id,
        "tmdb_cache_status": "missing_or_invalid_cache",
        "tmdb_title": None,
        "tmdb_original_title": None,
        "tmdb_overview": None,
        "tmdb_tagline": None,
        "tmdb_release_date": None,
        "tmdb_runtime": None,
        "tmdb_original_language": None,
        "tmdb_popularity": None,
        "tmdb_vote_average": None,
        "tmdb_vote_count": None,
        "tmdb_genres": None,
        "tmdb_keywords": None,
        "tmdb_directors": None,
        "tmdb_top_cast": None,
        "tmdb_production_companies": None,
    }

    if not cache_path.exists():
        tmdb_records.append(record)
        continue

    try:
        with cache_path.open("r", encoding="utf-8") as f:
            movie = json.load(f)

        # Apply the same essential cache validity checks.
        if (
            not isinstance(movie, dict)
            or movie.get("id") != tmdb_id
            or not isinstance(movie.get("credits"), dict)
            or not isinstance(movie.get("keywords"), dict)
        ):
            tmdb_records.append(record)
            continue

        credits = movie["credits"]
        keyword_data = movie["keywords"]

        keyword_items = (
            keyword_data.get("keywords")
            or keyword_data.get("results")
            or []
        )

        directors = [
            person["name"].strip()
            for person in credits.get("crew", [])
            if isinstance(person, dict)
            and person.get("job") == "Director"
            and isinstance(person.get("name"), str)
            and person["name"].strip()
        ]

        cast = get_names(credits.get("cast", [])[:10])

        genres = get_names(movie.get("genres", []))
        keywords = get_names(keyword_items)
        production_companies = get_names(
            movie.get("production_companies", [])
        )

        record.update({
            "tmdb_cache_status": "valid_cache",
            "tmdb_title": movie.get("title"),
            "tmdb_original_title": movie.get("original_title"),
            "tmdb_overview": movie.get("overview") or None,
            "tmdb_tagline": movie.get("tagline") or None,
            "tmdb_release_date": movie.get("release_date") or None,
            "tmdb_runtime": movie.get("runtime"),
            "tmdb_original_language": movie.get("original_language"),
            "tmdb_popularity": movie.get("popularity"),
            "tmdb_vote_average": movie.get("vote_average"),
            "tmdb_vote_count": movie.get("vote_count"),
            "tmdb_genres": " | ".join(genres) or None,
            "tmdb_keywords": " | ".join(keywords) or None,
            "tmdb_directors": " | ".join(directors) or None,
            "tmdb_top_cast": " | ".join(cast) or None,
            "tmdb_production_companies": (
                " | ".join(production_companies) or None
            ),
        })

    except (OSError, json.JSONDecodeError, TypeError, ValueError):
        # Preserve unresolved records; do not silently treat them as valid.
        pass

    tmdb_records.append(record)

tmdb_features = pd.DataFrame(tmdb_records)
tmdb_features["tmdbId_key"] = pd.array(
    tmdb_features["tmdbId_key"], dtype="Int64"
)

# Keep precisely one row per MovieLens catalogue item.
base_catalog = mapping[
    ["i", "movieId", "imdbId", "tmdbId", "title"]
].copy()

base_catalog["_tmdbId_key"] = (
    tmdb_numeric.where(valid_id_mask)
    .round()
    .astype("Int64")
)

# Multiple MovieLens titles may share a TMDB ID.
# This many-to-one join preserves every original catalogue row.
enriched_catalog = base_catalog.merge(
    tmdb_features,
    left_on="_tmdbId_key",
    right_on="tmdbId_key",
    how="left",
    validate="many_to_one",
)

enriched_catalog = enriched_catalog.drop(
    columns=["_tmdbId_key", "tmdbId_key"]
)

enriched_catalog["tmdb_cache_status"] = (
    enriched_catalog["tmdb_cache_status"]
    .fillna("missing_tmdb_id")
)

enriched_catalog["tmdb_release_year"] = (
    pd.to_datetime(
        enriched_catalog["tmdb_release_date"],
        errors="coerce",
    )
    .dt.year
    .astype("Int64")
)

# Convenience text field for later content-based experiments.
def make_tmdb_content_text(row):
    sections = [
        ("Overview", row["tmdb_overview"]),
        ("Genres", row["tmdb_genres"]),
        ("Keywords", row["tmdb_keywords"]),
        ("Director", row["tmdb_directors"]),
        ("Cast", row["tmdb_top_cast"]),
    ]

    return " ".join(
        f"{label}: {value}"
        for label, value in sections
        if isinstance(value, str) and value.strip()
    ) or None

enriched_catalog["tmdb_content_text"] = enriched_catalog.apply(
    make_tmdb_content_text, axis=1
)

# Safety checks before saving.
assert len(enriched_catalog) == len(mapping)
assert enriched_catalog["movieId"].is_unique
assert enriched_catalog["i"].is_unique

# Save the derived dataset; the original cache remains untouched.
OUTPUT_DIR = DATA_ROOT / "interim" / "sample_10pct"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "tmdb_enriched_catalog.parquet"

enriched_catalog.to_parquet(OUTPUT_PATH, index=False)

print("TMDB ENRICHMENT COMPLETE")
print("=" * 50)
print(f"Catalogue rows: {len(enriched_catalog):,}")
print(
    "Unique cached TMDB IDs:",
    tmdb_features["tmdb_cache_status"].eq("valid_cache").sum()
)
print("\nEnrichment status by catalogue row:")
display(
    enriched_catalog["tmdb_cache_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="rows")
)

print("\nMetadata completeness:")
for col in [
    "tmdb_overview",
    "tmdb_genres",
    "tmdb_keywords",
    "tmdb_directors",
    "tmdb_top_cast",
]:
    print(f"{col}: {enriched_catalog[col].notna().sum():,}")

print("\nSample enriched records:")
display(
    enriched_catalog[
        [
            "movieId",
            "title",
            "tmdb_title",
            "tmdb_genres",
            "tmdb_directors",
            "tmdb_cache_status",
        ]
    ].head(10)
)

print(f"\nSaved to: {OUTPUT_PATH}")

TMDB ENRICHMENT COMPLETE
Catalogue rows: 17,241
Unique cached TMDB IDs: 17036

Enrichment status by catalogue row:


,status,rows
0,valid_cache,17044
1,missing_or_invalid_cache,182
2,missing_tmdb_id,15



Metadata completeness:
tmdb_overview: 17,043
tmdb_genres: 17,042
tmdb_keywords: 16,207
tmdb_directors: 17,035
tmdb_top_cast: 16,977

Sample enriched records:


,movieId,title,tmdb_title,tmdb_genres,tmdb_directors,tmdb_cache_status
0,1,Toy Story (1995),Toy Story,Family | Comedy | Animation | Adventure,John Lasseter,valid_cache
1,2,Jumanji (1995),Jumanji,Adventure | Fantasy | Family,Joe Johnston,valid_cache
2,3,Grumpier Old Men (1995),Grumpier Old Men,Romance | Comedy,Howard Deutch,valid_cache
3,4,Waiting to Exhale (1995),Waiting to Exhale,Comedy | Drama | Romance,Forest Whitaker,valid_cache
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,Comedy | Family,Charles Shyer,valid_cache
5,6,Heat (1995),Heat,Crime | Drama | Action,Michael Mann,valid_cache
6,7,Sabrina (1995),Sabrina,Romance | Drama | Comedy,Sydney Pollack,valid_cache
7,8,Tom and Huck (1995),Tom and Huck,Family | Action | Adventure | Drama,Peter Hewitt,valid_cache
8,9,Sudden Death (1995),Sudden Death,Action | Drama | Thriller | Crime,Peter Hyams,valid_cache
9,10,GoldenEye (1995),GoldenEye,Adventure | Action | Thriller,Martin Campbell,valid_cache



Saved to: c:\Users\Lenovo\Documents\movie-recommender\data\interim\sample_10pct\tmdb_enriched_catalog.parquet


In [12]:

# TMDB enrichment quality audit — local files only

import re
import unicodedata
from difflib import SequenceMatcher

audit = pd.read_parquet(OUTPUT_PATH)

# 1. Check catalogue integrity
print("CATALOGUE INTEGRITY")
print("Rows:", len(audit))
print("Unique movieId:", audit["movieId"].nunique())
print("Unique internal index:", audit["i"].nunique())

assert audit["movieId"].is_unique, "Duplicate MovieLens movieId!"
assert audit["i"].is_unique, "Duplicate internal index!"

# 2. Compare MovieLens and TMDB titles
def normalize_title(value):
    if not isinstance(value, str) or not value.strip():
        return ""

    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        ch for ch in value
        if not unicodedata.combining(ch)
    )
    value = re.sub(r"\s*\(\d{4}\)\s*$", "", value)
    value = value.casefold()
    value = re.sub(r"[^a-z0-9]+", " ", value)

    return value.strip()

audit["ml_title_norm"] = audit["title"].map(normalize_title)
audit["tmdb_title_norm"] = audit["tmdb_title"].map(normalize_title)

audit["title_similarity"] = audit.apply(
    lambda r: SequenceMatcher(
        None,
        r["ml_title_norm"],
        r["tmdb_title_norm"],
    ).ratio()
    if r["ml_title_norm"] and r["tmdb_title_norm"]
    else float("nan"),
    axis=1,
)

valid = audit["tmdb_cache_status"].eq("valid_cache")

print("\nTITLE MATCHING — CACHED ROWS")
print("Cached catalogue rows:", int(valid.sum()))
print(
    "Exact normalized title matches:",
    int(
        (
            valid
            & audit["ml_title_norm"].eq(audit["tmdb_title_norm"])
            & audit["ml_title_norm"].ne("")
        ).sum()
    ),
)
print(
    "Low title similarity (< 0.70):",
    int((valid & audit["title_similarity"].lt(0.70)).sum()),
)

# Inspect potential mismatches; these need review, not automatic deletion.
print("\nPOTENTIAL TITLE MISMATCHES")
display(
    audit.loc[
        valid & audit["title_similarity"].lt(0.70),
        [
            "movieId",
            "title",
            "tmdb_title",
            "tmdbId",
            "title_similarity",
        ],
    ]
    .sort_values("title_similarity")
    .head(20)
)

# 3. Compare release years
ml_year = pd.to_numeric(
    audit["title"].str.extract(r"\((\d{4})\)\s*$")[0],
    errors="coerce",
)

tmdb_year = pd.to_numeric(
    audit["tmdb_release_date"].str[:4],
    errors="coerce",
)

both_years = valid & ml_year.notna() & tmdb_year.notna()
year_difference = (ml_year - tmdb_year).abs()

print("\nRELEASE YEAR CHECK")
print("Rows with both years available:", int(both_years.sum()))
print(
    "Year differences greater than 1:",
    int((both_years & year_difference.gt(1)).sum()),
)

display(
    audit.loc[
        both_years & year_difference.gt(1),
        [
            "movieId",
            "title",
            "tmdb_title",
            "tmdb_release_date",
            "tmdbId",
        ],
    ].head(20)
)

# 4. Metadata completeness among successfully cached rows
metadata_cols = [
    "tmdb_overview",
    "tmdb_genres",
    "tmdb_keywords",
    "tmdb_directors",
    "tmdb_top_cast",
]

print("\nMETADATA COMPLETENESS (%)")
display(
    (
        audit.loc[valid, metadata_cols]
        .notna()
        .mean()
        .mul(100)
        .round(2)
        .rename("present_pct")
        .reset_index(name="field")
    )
)


CATALOGUE INTEGRITY
Rows: 17241
Unique movieId: 17241
Unique internal index: 17241

TITLE MATCHING — CACHED ROWS
Cached catalogue rows: 17044
Exact normalized title matches: 12063
Low title similarity (< 0.70): 2799

POTENTIAL TITLE MISMATCHES


,movieId,title,tmdb_title,tmdbId,title_similarity
10210,67405,13B (2009),Yavarum Nalam,19950.0,0.000000
8126,31956,5x2 (2004),Five Times Two,4974.0,0.000000
12521,107537,Q (2011),Desire,86331.0,0.000000
14194,150993,Anacleto: Agente secreto (2015),Spy Time,284514.0,0.064516
7432,26002,Confidential Report (1955),Mr. Arkadin,44026.0,0.068966
14009,144374,Psychometry (2013),The Gifted Hands,193717.0,0.074074
13565,133869,MI-5 (2015),Spooks: The Greater Good,292040.0,0.074074
14179,150194,Quo vado? (2016),Where Am I Going?,374416.0,0.083333
14635,164455,Tschick (2016),Goodbye Berlin,384664.0,0.095238
5434,6120,Q: The Winged Serpent (1982),Q,27726.0,0.095238



RELEASE YEAR CHECK
Rows with both years available: 17006
Year differences greater than 1: 73


,movieId,title,tmdb_title,tmdb_release_date,tmdbId
117,121,"Boys of St. Vincent, The (1992)",The Boys of St. Vincent,1994-06-03,32119.0
700,746,Force of Evil (1948),Force of Evil,1950-08-24,26744.0
807,864,"Wife, The (1995)",Wife,1953-04-29,132641.0
830,895,Venice/Venice (1992),Venice,2010-06-11,79782.0
1021,1098,"Search for One-eye Jimmy, The (1996)",The Search for One-eye Jimmy,1994-11-09,67365.0
1036,1119,Drunks (1995),Drunks,1997-03-14,116844.0
1242,1349,Vampire in Venice (Nosferatu a Venezia) (Nosfe...,Nosferatu in Venice,1988-09-10,28198.0
1539,1692,Alien Escape (1995),Alien Escape,1997-12-16,29938.0
1711,1924,Plan 9 from Outer Space (1959),Plan 9 from Outer Space,1957-03-15,10513.0
1942,2158,"Henry: Portrait of a Serial Killer, Part 2 (1998)","Henry: Portrait of a Serial Killer, Part 2",1996-10-27,39930.0



METADATA COMPLETENESS (%)


,index,field
0,tmdb_overview,99.99
1,tmdb_genres,99.99
2,tmdb_keywords,95.09
3,tmdb_directors,99.95
4,tmdb_top_cast,99.61


In [13]:
# Refined TMDB mapping audit — no API calls

import pandas as pd
from difflib import SequenceMatcher
from pathlib import Path

# Compare MovieLens title against both TMDB title fields.
audit = pd.read_parquet(OUTPUT_PATH)

audit["ml_title_norm"] = audit["title"].map(normalize_title)
audit["tmdb_title_norm"] = audit["tmdb_title"].map(normalize_title)
audit["tmdb_original_title_norm"] = (
    audit["tmdb_original_title"].map(normalize_title)
)

def similarity(a, b):
    if not a or not b:
        return float("nan")
    return SequenceMatcher(None, a, b).ratio()

audit["display_title_similarity"] = audit.apply(
    lambda r: similarity(
        r["ml_title_norm"], r["tmdb_title_norm"]
    ), axis=1
)

audit["original_title_similarity"] = audit.apply(
    lambda r: similarity(
        r["ml_title_norm"], r["tmdb_original_title_norm"]
    ), axis=1
)

# A better preliminary score; still not a definitive identity check.
audit["best_title_similarity"] = audit[
    ["display_title_similarity", "original_title_similarity"]
].max(axis=1)

valid = audit["tmdb_cache_status"].eq("valid_cache")

audit["ml_year"] = pd.to_numeric(
    audit["title"].str.extract(r"\((\d{4})\)\s*$")[0],
    errors="coerce",
)

audit["tmdb_year"] = pd.to_numeric(
    audit["tmdb_release_date"].str[:4],
    errors="coerce",
)

audit["year_gap"] = (
    audit["ml_year"] - audit["tmdb_year"]
).abs()

exact_any = (
    audit["ml_title_norm"].eq(audit["tmdb_title_norm"])
    | audit["ml_title_norm"].eq(audit["tmdb_original_title_norm"])
) & audit["ml_title_norm"].ne("")

print("REFINED TITLE AUDIT")
print("Cached catalogue rows:", int(valid.sum()))
print(
    "Exact title match against either TMDB title:",
    int((valid & exact_any).sum())
)
print(
    "Best title similarity < 0.70:",
    int((valid & audit["best_title_similarity"].lt(0.70)).sum())
)
print(
    "Release year gap > 1:",
    int((valid & audit["year_gap"].gt(1)).sum())
)

# Save all year discrepancies for review.
review_cols = [
    "movieId",
    "title",
    "tmdb_title",
    "tmdb_original_title",
    "tmdb_release_date",
    "tmdbId",
    "display_title_similarity",
    "original_title_similarity",
    "best_title_similarity",
    "year_gap",
]

year_review = (
    audit.loc[valid & audit["year_gap"].gt(1), review_cols]
    .sort_values(
        ["year_gap", "best_title_similarity"],
        ascending=[False, True]
    )
)

output_dir = Path(OUTPUT_PATH).parent

year_review.to_csv(
    output_dir / "tmdb_year_mismatch_review.csv",
    index=False,
)

# Prioritize rows where BOTH title similarity and year disagree.
priority_review = year_review.loc[
    year_review["best_title_similarity"].lt(0.50)
].copy()

priority_review.to_csv(
    output_dir / "tmdb_high_priority_mapping_review.csv",
    index=False,
)

print("\nHighest-priority candidates:")
display(priority_review.head(30))

print("\nAll release-year discrepancies — first 20:")
display(year_review.head(20))

print("\nSaved review files to:", output_dir)

REFINED TITLE AUDIT
Cached catalogue rows: 17044
Exact title match against either TMDB title: 12144
Best title similarity < 0.70: 2502
Release year gap > 1: 73

Highest-priority candidates:


,movieId,title,tmdb_title,tmdb_original_title,tmdb_release_date,tmdbId,display_title_similarity,original_title_similarity,best_title_similarity,year_gap
14001,144172,February (2015),The Blackcoat's Daughter,The Blackcoat's Daughter,2017-03-31,334536.0,0.187500,0.187500,0.187500,2.0
5113,5724,"Creature Wasn't Nice, The (a.k.a. Naked Space)...",Spaceship,Spaceship,1983-07-01,36220.0,0.268657,0.268657,0.268657,2.0
12215,101747,"War of the Gargantuas, The (Furankenshutain no...",The War of the Gargantuas,フランケンシュタインの怪獣 サンダ対ガイラ,1966-07-31,3146.0,0.461538,NaN,0.461538,2.0
8099,31702,Turtles Can Fly (Lakposhtha hâm parvaz mikonan...,Turtles Can Fly,کیسەڵەکانیش دەفڕن,2006-12-28,8340.0,0.491803,NaN,0.491803,2.0



All release-year discrepancies — first 20:


,movieId,title,tmdb_title,tmdb_original_title,tmdb_release_date,tmdbId,display_title_similarity,original_title_similarity,best_title_similarity,year_gap
13226,126426,Solyaris (1968),Solaris,Солярис,2019-04-14,321130.0,0.933333,NaN,0.933333,51.0
807,864,"Wife, The (1995)",Wife,妻,1953-04-29,132641.0,0.666667,NaN,0.666667,42.0
9133,50740,Seven Up! (1964),Seven Up!,Seven Up!,1985-10-06,51863.0,1.000000,1.000000,1.000000,21.0
830,895,Venice/Venice (1992),Venice,Wenecja,2010-06-11,79782.0,0.631579,0.300000,0.631579,18.0
3816,4249,Kingdom Come (2001),Kingdom Come,Kingdom Come,2012-04-28,136558.0,1.000000,1.000000,1.000000,11.0
10422,70871,"Day in the Country, A (Partie de campagne) (1936)",A Day in the Country,Partie de campagne,1946-05-21,43878.0,0.610169,0.631579,0.631579,10.0
5696,6430,Ziggy Stardust and the Spiders from Mars (1973),Ziggy Stardust and the Spiders from Mars,Ziggy Stardust and the Spiders from Mars,1983-12-23,34759.0,1.000000,1.000000,1.000000,10.0
15697,189957,The Witness (1969),The Witness,A tanú,1979-06-06,40987.0,1.000000,0.235294,1.000000,10.0
10799,76823,"Men Who Tread on the Tiger's Tail, The (Tora n...",The Men Who Tread on the Tiger's Tail,虎の尾を踏む男達,1952-04-24,36916.0,0.640777,NaN,0.640777,7.0
14808,167770,Christmas Time In South Park (2000),Christmas Time in South Park,Christmas Time in South Park,2007-11-13,148039.0,1.000000,1.000000,1.000000,7.0



Saved review files to: c:\Users\Lenovo\Documents\movie-recommender\data\interim\sample_10pct


In [14]:

# Cross-check MovieLens IMDb IDs against cached TMDB IMDb IDs.
# LOCAL FILES ONLY — no API requests.

import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

TMDB_CACHE_DIR = DATA_ROOT / "raw" / "tmdb_api_cache"

def normalize_imdb_id(value):
    """Normalize MovieLens numeric IMDb IDs and TMDB tt-prefixed IDs."""
    if pd.isna(value):
        return None

    s = str(value).strip().lower()

    if not s or s in {"0", "0.0", "nan", "none"}:
        return None

    if s.startswith("tt"):
        digits = re.sub(r"\D", "", s[2:])
    else:
        try:
            digits = str(int(float(s)))
        except (ValueError, OverflowError):
            digits = re.sub(r"\D", "", s)

    if not digits or int(digits) == 0:
        return None

    return "tt" + digits.zfill(7)


# One audit record per unique TMDB ID.
numeric_tmdb = pd.to_numeric(mapping["tmdbId"], errors="coerce")

valid_tmdb = (
    numeric_tmdb.notna()
    & numeric_tmdb.gt(0)
    & np.isclose(
        numeric_tmdb.fillna(0),
        numeric_tmdb.fillna(0).round(),
    )
)

unique_ids = sorted(
    numeric_tmdb.loc[valid_tmdb].astype("int64").unique()
)

identity_rows = []

for tmdb_id in unique_ids:
    path = TMDB_CACHE_DIR / f"{tmdb_id}.json"

    row = {
        "tmdbId_key": int(tmdb_id),
        "cache_found": False,
        "cached_imdbId": None,
        "cached_tmdb_title": None,
        "cached_tmdb_release_date": None,
    }

    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        if (
            isinstance(payload, dict)
            and payload.get("id") == int(tmdb_id)
            and isinstance(payload.get("credits"), dict)
            and isinstance(payload.get("keywords"), dict)
        ):
            row.update({
                "cache_found": True,
                "cached_imdbId": normalize_imdb_id(
                    payload.get("imdb_id")
                ),
                "cached_tmdb_title": payload.get("title"),
                "cached_tmdb_release_date": payload.get(
                    "release_date"
                ),
            })

    except (OSError, json.JSONDecodeError, TypeError, ValueError):
        pass

    identity_rows.append(row)

tmdb_identity = pd.DataFrame(identity_rows)

# Compare identity fields without modifying the original mapping.
id_audit = mapping[
    ["i", "movieId", "title", "imdbId", "tmdbId"]
].copy()

id_audit["tmdbId_key"] = (
    pd.to_numeric(id_audit["tmdbId"], errors="coerce")
    .round()
    .astype("Int64")
)

id_audit["movielens_imdbId"] = id_audit["imdbId"].map(
    normalize_imdb_id
)

id_audit = id_audit.merge(
    tmdb_identity,
    on="tmdbId_key",
    how="left",
    validate="many_to_one",
)

id_audit["imdb_check_status"] = "unverified"

no_cache = id_audit["cache_found"].ne(True)

id_audit.loc[no_cache, "imdb_check_status"] = "no_valid_cache"

has_cache = ~no_cache
missing_ml_imdb = id_audit["movielens_imdbId"].isna()
missing_tmdb_imdb = id_audit["cached_imdbId"].isna()

id_audit.loc[
    has_cache & missing_ml_imdb,
    "imdb_check_status"
] = "missing_movielens_imdbId"

id_audit.loc[
    has_cache & ~missing_ml_imdb & missing_tmdb_imdb,
    "imdb_check_status"
] = "missing_tmdb_imdbId"

comparable = (
    has_cache
    & ~missing_ml_imdb
    & ~missing_tmdb_imdb
)

id_audit.loc[
    comparable
    & id_audit["movielens_imdbId"].eq(id_audit["cached_imdbId"]),
    "imdb_check_status"
] = "MATCH"

id_audit.loc[
    comparable
    & id_audit["movielens_imdbId"].ne(id_audit["cached_imdbId"]),
    "imdb_check_status"
] = "MISMATCH"

print("IMDb ID CROSS-CHECK")
display(
    id_audit["imdb_check_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="catalogue_rows")
)

mismatches = (
    id_audit.loc[
        id_audit["imdb_check_status"].eq("MISMATCH"),
        [
            "movieId",
            "title",
            "imdbId",
            "movielens_imdbId",
            "tmdbId",
            "cached_imdbId",
            "cached_tmdb_title",
            "cached_tmdb_release_date",
        ],
    ]
    .sort_values(["movieId"])
)

print("\nIDENTIFIER MISMATCHES — PRIORITY REVIEW")
display(mismatches.head(50))

# Save a review artifact, not a modified catalogue.
review_path = (
    DATA_ROOT / "interim" / "sample_10pct"
    / "tmdb_imdb_id_mismatch_review.csv"
)
review_path.parent.mkdir(parents=True, exist_ok=True)
mismatches.to_csv(review_path, index=False)

print(f"\nFull mismatch report saved to: {review_path}")


IMDb ID CROSS-CHECK


,status,catalogue_rows
0,MATCH,17001
1,no_valid_cache,197
2,MISMATCH,42
3,missing_tmdb_imdbId,1



IDENTIFIER MISMATCHES — PRIORITY REVIEW


,movieId,title,imdbId,movielens_imdbId,tmdbId,cached_imdbId,cached_tmdb_title,cached_tmdb_release_date
677,720,Wallace & Gromit: The Best of Aardman Animatio...,118114,tt0118114,503475.0,tt31416047,Wallace & Gromit: The Best of Aardman Animation,1996-06-14
807,864,"Wife, The (1995)",114936,tt0114936,132641.0,tt0046468,Wife,1953-04-29
830,895,Venice/Venice (1992),105729,tt0105729,79782.0,tt1684935,Venice,2010-06-11
2126,2365,King Kong vs. Godzilla (Kingukongu tai Gojira)...,56142,tt0056142,1680.0,tt13364334,King Kong vs. Godzilla,1962-08-11
2265,2510,Just the Ticket (1999),134948,tt0134948,26425.0,tt0123221,Just the Ticket,1998-02-01
3808,4241,Pokémon 3: The Movie (2001),266860,tt0266860,10991.0,tt0235679,Pokémon 3: The Movie,2000-07-08
3816,4249,Kingdom Come (2001),246002,tt0246002,136558.0,tt2043893,Kingdom Come,2012-04-28
4725,5264,Clockstoppers (2002),250305,tt0250305,15028.0,tt0157472,Clockstoppers,2002-03-29
5074,5672,Pokemon 4 Ever (a.k.a. Pokémon 4: The Movie) (...,313487,tt0313487,12600.0,tt0287635,Pokémon 4Ever,2001-07-07
5343,6003,Confessions of a Dangerous Mind (2002),290538,tt0290538,4912.0,tt0270288,Confessions of a Dangerous Mind,2002-12-31



Full mismatch report saved to: c:\Users\Lenovo\Documents\movie-recommender\data\interim\sample_10pct\tmdb_imdb_id_mismatch_review.csv
